## 0 — Imports & device

In [ ]:
# !pip install torch numpy matplotlib scipy python-docx --break-system-packages
import os, re, time, csv, datetime, json as _json, tempfile, shutil, warnings, importlib, subprocess, sys
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
import matplotlib; matplotlib.use('inline')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE.upper()}')


## 1 — Run folder (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/BeamTransformer_Comparison'

def _next_run_id(base):
    existing = [d for d in os.listdir(base)
                if os.path.isdir(os.path.join(base,d)) and re.match(r'^Run_\d+$',d)] \
               if os.path.isdir(base) else []
    ids = [int(re.search(r'\d+',d).group()) for d in existing]
    return max(ids, default=0) + 1

os.makedirs(BASE_DIR, exist_ok=True)
RUN_ID  = _next_run_id(BASE_DIR)
RUN_DIR = os.path.join(BASE_DIR, f'Run_{RUN_ID}')
CHECKPOINT_DIR = os.path.join(RUN_DIR, 'checkpoints')
RESULTS_DIR    = os.path.join(RUN_DIR, 'results')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)

# ── Shared dataset cache from beamforming_project_colab_v4_WMMSE.ipynb ──
# That notebook's Drive-mount cell defines:
#   BASE_DIR (v4)          = '/content/drive/MyDrive/BeamTransformer'
#   DATASET_CACHE_DIR (v4) = BASE_DIR/dataset_cache
# This is a DIFFERENT project folder from this notebook's own BASE_DIR
# (BeamTransformer_Comparison) — we point at it explicitly so this
# notebook's get_dataloaders() call can hit the exact same config-hashed
# cache subfolder (chunks + val/test .npz + offline WMMSE labels) that v4
# already built, instead of regenerating everything from scratch.
V4_DATASET_CACHE_DIR = '/content/drive/MyDrive/BeamTransformer_Comparison/dataset_cache'

print(f'Run folder         : {RUN_DIR}')
print(f'V4 dataset cache   : {V4_DATASET_CACHE_DIR}')


## 2 — Channel simulation

In [ ]:
def ula_steering(theta: np.ndarray, Nt: int) -> np.ndarray:
    """ULA steering vector. theta shape (...,); returns (..., Nt)."""
    n = np.arange(Nt)
    phase = np.pi * np.sin(theta)[..., np.newaxis] * n
    return (1.0 / np.sqrt(Nt)) * np.exp(1j * phase)


def upa_steering(theta: np.ndarray, phi: np.ndarray, Nv: int, Nh: int) -> np.ndarray:
    """UPA steering vector. Returns (..., Nv*Nh)."""
    av = ula_steering(theta, Nv)
    ah = ula_steering(phi,   Nh)
    a  = np.einsum('...i,...j->...ij', av, ah)
    return a.reshape(*a.shape[:-2], Nv * Nh)


class MassiveMIMOChannel:
    """
    Geometry-based stochastic channel model. Same implementation used in
    beamforming_project_colab_v4_WMMSE.ipynb (kept identical so the two
    notebooks' dataset configs hash the same way for cache reuse).
    Nt: TX antennas, K: users, L: multipath per user,
    array_type: 'ULA' or 'UPA', ang_spread: degrees.
    """
    def __init__(self, Nt=32, K=8, L=5, array_type='ULA',
                 Nv=4, Nh=8, ang_spread=15.0):
        assert array_type in ('ULA', 'UPA')
        if array_type == 'UPA':
            assert Nv * Nh == Nt
        self.Nt, self.K, self.L = Nt, K, L
        self.array_type = array_type
        self.Nv, self.Nh = Nv, Nh
        self.ang_spread = np.deg2rad(ang_spread)

    def generate(self, batch_size: int, rng=None):
        """Return complex channel H of shape (batch_size, K, Nt)."""
        if rng is None:
            rng = np.random.default_rng()
        Nt, K, L, B = self.Nt, self.K, self.L, batch_size

        theta_mean = rng.uniform(-np.pi / 3, np.pi / 3, size=(B, K))
        delta = rng.uniform(-self.ang_spread / 2, self.ang_spread / 2, size=(B, K, L))
        theta = theta_mean[..., np.newaxis] + delta

        alpha = (rng.standard_normal((B, K, L)) +
                 1j * rng.standard_normal((B, K, L))) / np.sqrt(2)

        if self.array_type == 'ULA':
            a = ula_steering(theta, Nt)
        else:
            phi_mean = rng.uniform(-np.pi / 3, np.pi / 3, size=(B, K))
            phi = phi_mean[..., np.newaxis] + delta
            a   = upa_steering(theta, phi, self.Nv, self.Nh)

        H = np.einsum('bkl,bkln->bkn', alpha, a) / np.sqrt(L)
        H = H / np.sqrt(np.mean(np.abs(H)**2, axis=-1, keepdims=True) * Nt)
        return H


def add_csi_noise(H: np.ndarray, pilot_snr_db: float, rng=None) -> np.ndarray:
    """H_hat = H + E,  E ~ CN(0, sigma_e^2 I),  sigma_e^2 = 1/(1+SNR_p)."""
    if rng is None:
        rng = np.random.default_rng()
    pilot_snr_lin = 10 ** (pilot_snr_db / 10)
    sigma_e = np.sqrt(1.0 / (1.0 + pilot_snr_lin))
    noise = sigma_e * (rng.standard_normal(H.shape) +
                       1j * rng.standard_normal(H.shape)) / np.sqrt(2)
    return H + noise


## 3 — Dataset (chunked, low-RAM)

In [ ]:
import hashlib, json as _json


def _dataset_config_hash(cfg: dict) -> str:
    s = _json.dumps(cfg, sort_keys=True, default=str)
    return hashlib.md5(s.encode()).hexdigest()[:10]


class MIMODataset(Dataset):
    """
    Pre-generated dataset of (H_hat, H, snr_db, W_wmmse) tuples.
    Channels stored as concatenated real/imag float32 vectors.
    W_wmmse is the offline WMMSE pre-training target, computed from H_hat
    (not the perfect channel), flattened with the same convention BeamNet
    uses internally (real block then imag block, Nt-major / K-minor).
    """
    def __init__(self, H_hat: np.ndarray, H: np.ndarray, snr_db: np.ndarray,
                 W_wmmse: np.ndarray):
        super().__init__()
        B, K, Nt = H.shape
        def to_real(X):
            return np.concatenate([X.real, X.imag], axis=-1).reshape(B, -1).astype(np.float32)
        self.H_hat   = torch.from_numpy(to_real(H_hat))
        self.H       = torch.from_numpy(to_real(H))
        self.snr     = torch.from_numpy(snr_db.astype(np.float32))
        self.W_wmmse = torch.from_numpy(W_wmmse.astype(np.float32))
        self.K, self.Nt = K, Nt

    def __len__(self):  return len(self.snr)
    def __getitem__(self, idx):
        return self.H_hat[idx], self.H[idx], self.snr[idx], self.W_wmmse[idx]

    def save_npz(self, path):
        np.savez(path, H_hat=self.H_hat.numpy(), H=self.H.numpy(),
                 snr=self.snr.numpy(), W_wmmse=self.W_wmmse.numpy(),
                 K=self.K, Nt=self.Nt)

    @classmethod
    def load_npz(cls, path):
        d = np.load(path)
        obj = cls.__new__(cls)
        obj.H_hat   = torch.from_numpy(d['H_hat'])
        obj.H       = torch.from_numpy(d['H'])
        obj.snr     = torch.from_numpy(d['snr'])
        obj.W_wmmse = torch.from_numpy(d['W_wmmse'])
        obj.K, obj.Nt = int(d['K']), int(d['Nt'])
        return obj


class ChunkedMIMODataset(Dataset):
    """
    Generates the dataset in small chunks and stores each chunk as a
    .npz file on disk (H_hat, H, snr, W_wmmse). Only one chunk is loaded
    into RAM at a time during training, keeping peak memory flat regardless
    of total dataset size.

    If `cache_dir` is given (e.g. the SHARED cache built by
    beamforming_project_colab_v4_WMMSE.ipynb) and it already holds a
    complete, matching set of chunks + manifest, generation — including the
    expensive per-sample WMMSE solve used for pre-training labels — is
    skipped entirely and the cached chunks are reused as-is. If `cache_dir`
    is None, a throwaway local temp directory is used instead.
    """
    def __init__(self, n_samples, channel, chunk_size=10000,
                 snr_range=(0, 30), pilot_snr_range=(0, 20), seed=0,
                 wmmse_n_iter=30, wmmse_tol=1e-4,
                 cache_dir=None, force_regenerate=False):
        super().__init__()
        self.n_samples  = n_samples
        self.chunk_size = chunk_size
        self.K, self.Nt = channel.K, channel.Nt
        self._persistent = cache_dir is not None

        if cache_dir is not None:
            os.makedirs(cache_dir, exist_ok=True)
            self._cache_dir = cache_dir
        else:
            self._cache_dir = tempfile.mkdtemp(prefix='mimo_ds_')

        manifest_path = os.path.join(self._cache_dir, 'manifest.json')
        expected_config = {
            'n_samples': n_samples, 'chunk_size': chunk_size,
            'K': self.K, 'Nt': self.Nt,
            'snr_range': list(snr_range), 'pilot_snr_range': list(pilot_snr_range),
            'seed': seed, 'wmmse_n_iter': wmmse_n_iter, 'wmmse_tol': wmmse_tol,
        }

        cached_ok = (not force_regenerate) and os.path.exists(manifest_path)
        saved_manifest = None
        if cached_ok:
            with open(manifest_path) as f:
                saved_manifest = _json.load(f)
            cached_ok = (saved_manifest.get('config') == expected_config and
                         all(os.path.exists(p) for p in saved_manifest.get('chunk_files', [])))

        if cached_ok:
            print(f'[Dataset] Reusing cached chunks at {self._cache_dir}')
            self._chunk_files = saved_manifest['chunk_files']
            self._chunk_sizes = saved_manifest['chunk_sizes']
        else:
            self._chunk_files = []
            self._chunk_sizes = []

            rng = np.random.default_rng(seed)
            n_chunks = int(np.ceil(n_samples / chunk_size))
            written  = 0

            print(f'  Writing {n_chunks} chunks to {self._cache_dir} ...', end=' ', flush=True)
            for c in range(n_chunks):
                size = min(chunk_size, n_samples - written)
                H    = channel.generate(size, rng=rng)
                snr_db    = rng.uniform(*snr_range,       size=size).astype(np.float32)
                pilot_snr = rng.uniform(*pilot_snr_range, size=size)
                pilot_snr_lin = 10 ** (pilot_snr / 10)
                sigma_e = np.sqrt(1.0 / (1.0 + pilot_snr_lin))[:, None, None]
                noise   = sigma_e * (rng.standard_normal(H.shape) +
                                     1j * rng.standard_normal(H.shape)) / np.sqrt(2)
                H_hat = H + noise

                W_wmmse = wmmse_targets_batch(H_hat, snr_db,
                                               n_iter=wmmse_n_iter, tol=wmmse_tol)

                def to_real(X):
                    B = X.shape[0]
                    return np.concatenate([X.real, X.imag], axis=-1).reshape(B, -1).astype(np.float32)

                path = os.path.join(self._cache_dir, f'chunk_{c:04d}.npz')
                np.savez(path, H_hat=to_real(H_hat), H=to_real(H),
                         snr=snr_db, W_wmmse=W_wmmse)
                self._chunk_files.append(path)
                self._chunk_sizes.append(size)
                written += size
                print(f'{c+1}', end=' ', flush=True)
            print('done.')

            with open(manifest_path, 'w') as f:
                _json.dump({'config': expected_config,
                            'chunk_files': self._chunk_files,
                            'chunk_sizes': self._chunk_sizes}, f)

        self._index_map = []
        for c, sz in enumerate(self._chunk_sizes):
            for i in range(sz):
                self._index_map.append((c, i))

        self._loaded_chunk_idx  = None
        self._loaded_chunk_data = None

    def _load_chunk(self, chunk_idx):
        if self._loaded_chunk_idx != chunk_idx:
            data = np.load(self._chunk_files[chunk_idx])
            self._loaded_chunk_data = (
                torch.from_numpy(data['H_hat']),
                torch.from_numpy(data['H']),
                torch.from_numpy(data['snr']),
                torch.from_numpy(data['W_wmmse']),
            )
            self._loaded_chunk_idx = chunk_idx

    def __len__(self): return self.n_samples

    def __getitem__(self, idx):
        chunk_idx, row = self._index_map[idx]
        self._load_chunk(chunk_idx)
        H_hat, H, snr, W_wmmse = self._loaded_chunk_data
        return H_hat[row], H[row], snr[row], W_wmmse[row]

    def cleanup(self):
        """Deletes the chunk cache — but only if it's a local throwaway
        temp dir. A shared/persistent cache (e.g. v4's Drive folder) is
        left untouched."""
        if self._persistent:
            print(f'[Dataset] Cache is persistent at {self._cache_dir} — not deleting.')
            return
        shutil.rmtree(self._cache_dir, ignore_errors=True)


class ChunkSequentialSampler(torch.utils.data.Sampler):
    """Yields indices chunk-by-chunk in shuffled order, never crossing
    chunk boundaries mid-batch, so only one chunk needs to be RAM-resident."""
    def __init__(self, dataset, shuffle=True, seed=0):
        self.dataset = dataset
        self.shuffle = shuffle
        self.seed    = seed
        self.epoch   = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.dataset)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        chunk_sizes = self.dataset._chunk_sizes
        chunk_order = rng.permutation(len(chunk_sizes)).tolist()
        offsets = []
        offset  = 0
        for sz in chunk_sizes:
            offsets.append(offset)
            offset += sz
        indices = []
        for c in chunk_order:
            start = offsets[c]
            sz    = chunk_sizes[c]
            chunk_indices = np.arange(start, start + sz)
            if self.shuffle:
                rng.shuffle(chunk_indices)
            indices.extend(chunk_indices.tolist())
        return iter(indices)


def generate_dataset(n_samples, channel, snr_range=(0, 30),
                     pilot_snr_range=(0, 20), seed=42,
                     wmmse_n_iter=30, wmmse_tol=1e-4,
                     cache_path=None, force_regenerate=False):
    """Builds an in-memory MIMODataset (used for val/test). Loads from
    cache_path (a single .npz, e.g. on Drive) if it already exists."""
    if cache_path is not None and os.path.exists(cache_path) and not force_regenerate:
        print(f'[Dataset] Loading cached dataset from {cache_path}')
        return MIMODataset.load_npz(cache_path)

    rng = np.random.default_rng(seed)
    H   = channel.generate(n_samples, rng=rng)
    snr_db    = rng.uniform(*snr_range,       size=n_samples)
    pilot_snr = rng.uniform(*pilot_snr_range, size=n_samples)
    pilot_snr_lin = 10 ** (pilot_snr / 10)
    sigma_e = np.sqrt(1.0 / (1.0 + pilot_snr_lin))[:, None, None]
    noise   = sigma_e * (rng.standard_normal(H.shape) +
                         1j * rng.standard_normal(H.shape)) / np.sqrt(2)
    H_hat   = H + noise
    W_wmmse = wmmse_targets_batch(H_hat, snr_db, n_iter=wmmse_n_iter, tol=wmmse_tol)
    ds = MIMODataset(H_hat, H, snr_db, W_wmmse)

    if cache_path is not None:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        ds.save_npz(cache_path)
        print(f'[Dataset] Saved to {cache_path}')
    return ds


def get_dataloaders(Nt, K, L, n_train, n_val, n_test, batch_size,
                    chunk_size=10000, array_type='ULA',
                    snr_range=(0, 30), pilot_snr_range=(0, 20),
                    wmmse_n_iter=30, wmmse_tol=1e-4,
                    cache_dir=None, force_regenerate=False):
    """
    Build train / val / test DataLoaders.

    cache_dir : optional path (e.g. V4_DATASET_CACHE_DIR from the Drive-mount
    cell). When given, the dataset is looked up under a config-hashed
    subfolder (Nt/K/L/array_type/dataset sizes/SNR ranges/WMMSE settings)
    and reused verbatim if it matches — this is how this notebook reuses
    the dataset (chunks + WMMSE labels) already built by
    beamforming_project_colab_v4_WMMSE.ipynb, instead of regenerating and
    re-solving WMMSE. If nothing matches, it's generated fresh and cached
    for next time. Pass force_regenerate=True to rebuild regardless.
    """
    channel = MassiveMIMOChannel(Nt=Nt, K=K, L=L, array_type=array_type)

    run_cache_dir = None
    if cache_dir is not None:
        cfg = {
            'Nt': Nt, 'K': K, 'L': L, 'array_type': array_type,
            'n_train': n_train, 'n_val': n_val, 'n_test': n_test,
            'snr_range': list(snr_range), 'pilot_snr_range': list(pilot_snr_range),
            'chunk_size': chunk_size,
            'wmmse_n_iter': wmmse_n_iter, 'wmmse_tol': wmmse_tol,
        }
        cfg_hash = _dataset_config_hash(cfg)
        run_cache_dir = os.path.join(cache_dir, f'cfg_{cfg_hash}')
        os.makedirs(run_cache_dir, exist_ok=True)
        print(f'[Data] Cache dir: {run_cache_dir}')

    train_cache = os.path.join(run_cache_dir, 'train') if run_cache_dir else None
    print(f'[Data] Train chunked ({n_train}) ...')
    tr_ds = ChunkedMIMODataset(n_train, channel, chunk_size=chunk_size,
                               snr_range=snr_range, pilot_snr_range=pilot_snr_range,
                               seed=0, wmmse_n_iter=wmmse_n_iter, wmmse_tol=wmmse_tol,
                               cache_dir=train_cache, force_regenerate=force_regenerate)
    tr_sm = ChunkSequentialSampler(tr_ds, shuffle=True, seed=0)

    val_cache  = os.path.join(run_cache_dir, 'val.npz')  if run_cache_dir else None
    test_cache = os.path.join(run_cache_dir, 'test.npz') if run_cache_dir else None
    print(f'[Data] Val ({n_val}) / Test ({n_test}) ...')
    va_ds = generate_dataset(n_val, channel, snr_range=snr_range, pilot_snr_range=pilot_snr_range,
                              seed=1, wmmse_n_iter=wmmse_n_iter, wmmse_tol=wmmse_tol,
                              cache_path=val_cache, force_regenerate=force_regenerate)
    te_ds = generate_dataset(n_test, channel, snr_range=snr_range, pilot_snr_range=pilot_snr_range,
                              seed=2, wmmse_n_iter=wmmse_n_iter, wmmse_tol=wmmse_tol,
                              cache_path=test_cache, force_regenerate=force_regenerate)

    kw = dict(num_workers=0, pin_memory=torch.cuda.is_available())
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, sampler=tr_sm, **kw)
    va_ld = DataLoader(va_ds, batch_size=batch_size, shuffle=False, **kw)
    te_ld = DataLoader(te_ds, batch_size=batch_size, shuffle=False, **kw)
    print(f'[Data] Done. Input dim={tr_ds.K*tr_ds.Nt*2}')
    return tr_ld, va_ld, te_ld, channel, tr_ds, tr_sm


## 4 — Classical beamformers & metrics

In [ ]:
def power_norm(W,P=1.0):
    n=np.linalg.norm(W,'fro'); return W if n<1e-12 else W*np.sqrt(P)/n
def mrt(H,P=1.0):   return power_norm(H.conj().T,P)
def zf_bf(H,P=1.0): return power_norm(H.conj().T@np.linalg.pinv(H@H.conj().T),P)
def mmse_bf(H,snr_db,P=1.0):
    K=H.shape[0]; s=10**(snr_db/10)
    return power_norm(H.conj().T@np.linalg.inv(H@H.conj().T+(K/s)*np.eye(K)),P)

def wmmse_bf(H, snr_db, P=1.0, n_iter=100, tol=1e-6):
    """
    Iterative WMMSE precoder (Shi, Razaviyayn, Luo & He, IEEE TSP 2011).
    Same interface/convention as beamforming_project_colab_v4_WMMSE.ipynb.
    Near-optimal reference baseline, and the source of the offline
    pre-training labels used below.
    """
    K, Nt = H.shape
    s = 10 ** (snr_db / 10)
    sigma2 = 1.0 / s
    V = power_norm(H.conj().T @ np.linalg.inv(H @ H.conj().T + (K / s) * np.eye(K)), P)
    prev_rate = -np.inf
    for _ in range(n_iter):
        HV  = H @ V
        hvk = np.diag(HV)
        u   = hvk / (np.sum(np.abs(HV) ** 2, axis=1) + sigma2)
        e   = 1.0 - u.conj() * hvk
        w   = 1.0 / np.real(e).clip(min=1e-10)
        diag_term = sigma2 * np.sum(w * np.abs(u) ** 2)
        A   = diag_term * np.eye(Nt) + (H.conj().T * (w * np.abs(u) ** 2)) @ H
        rhs = H.conj().T * (w * u)
        V   = np.linalg.solve(A + 1e-10 * np.eye(Nt), rhs)
        V   = power_norm(V, P)
        rate = compute_sum_rate(H, V, snr_db)
        if abs(rate - prev_rate) < tol:
            break
        prev_rate = rate
    return V

def compute_sinr(H,W,snr_db):
    K=H.shape[0]; s=10**(snr_db/10); HW=H@W; sinr=np.zeros(K)
    for k in range(K):
        sig=np.abs(HW[k,k])**2; tot=np.sum(np.abs(HW[k,:])**2)
        sinr[k]=sig/(tot-sig+1/s)
    return sinr
def compute_sum_rate(H,W,snr_db):
    return float(np.sum(np.log2(1+compute_sinr(H,W,snr_db))))
def compute_ber(H,W,snr_db,ns=5000,rng=None):
    if rng is None: rng=np.random.default_rng(0)
    K=H.shape[0]; s=10**(snr_db/10)
    b=rng.integers(0,2,(K,ns,2)); sy=(2*b-1).astype(complex)
    tx=(sy[...,0]+1j*sy[...,1])/np.sqrt(2); HW=H@W
    std=np.sqrt(1/(2*s))
    y=HW@tx+std*(rng.standard_normal((K,ns))+1j*rng.standard_normal((K,ns)))
    try: ye=np.linalg.solve(HW,y)
    except: ye=np.linalg.lstsq(HW,y,rcond=None)[0]
    bh=np.zeros_like(b); bh[...,0]=(ye.real>0).astype(int); bh[...,1]=(ye.imag>0).astype(int)
    return float(np.sum(bh!=b)/b.size)

def flat_to_complex_W(w,Nt,K):
    half=Nt*K; return w[:half].reshape(Nt,K)+1j*w[half:].reshape(Nt,K)


# ── WMMSE targets for supervised pre-training ─────────────────────────
# Same helper as in beamforming_project_colab_v4_WMMSE.ipynb: wmmse_bf() is
# an iterative per-sample numpy routine with no cheap batched/GPU form, so
# labels are generated once (in a process pool) and cached with the dataset.
from concurrent.futures import ProcessPoolExecutor
import os as _os

def _wmmse_flat_single(args):
    H_hat_k, snr, n_iter, tol = args
    W = wmmse_bf(H_hat_k, snr, P=1.0, n_iter=n_iter, tol=tol)   # (Nt, K) complex
    return np.concatenate([W.real.ravel(), W.imag.ravel()]).astype(np.float32)

def wmmse_targets_batch(H_hat_complex: np.ndarray, snr_db: np.ndarray,
                         n_iter: int = 30, tol: float = 1e-4,
                         n_workers: int = None) -> np.ndarray:
    """H_hat_complex: (B,K,Nt) complex, snr_db: (B,). Returns (B, 2*Nt*K) float32."""
    n_workers = n_workers or max(1, (_os.cpu_count() or 2) - 1)
    args = [(H_hat_complex[i], float(snr_db[i]), n_iter, tol) for i in range(len(snr_db))]
    if n_workers == 1 or len(args) < 256:
        return np.stack([_wmmse_flat_single(a) for a in args], axis=0)
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        out = list(ex.map(_wmmse_flat_single, args, chunksize=64))
    return np.stack(out, axis=0)


## 5 — All models (BeamTransformer Large/Small + 3 baselines)

In [ ]:
# ── Shared loss & eval ────────────────────────────────────────────────
class PowerNorm(nn.Module):
    def __init__(self,P=1.0): super().__init__(); self.P=P
    def forward(self,x):
        return x*(self.P**0.5)/torch.linalg.norm(x,dim=1,keepdim=True).clamp(min=1e-8)

def sinr_sumrate_loss(W_flat,H_flat,snr_db,Nt,K):
    B=W_flat.shape[0]
    W=torch.complex(W_flat[:,:Nt*K].view(B,Nt,K),W_flat[:,Nt*K:].view(B,Nt,K))
    H=torch.complex(H_flat[:,:K*Nt].view(B,K,Nt),H_flat[:,K*Nt:].view(B,K,Nt))
    HW=H@W; nl=1/10**(snr_db/10)
    sig=torch.abs(torch.diagonal(HW,dim1=1,dim2=2))**2
    tot=(torch.abs(HW)**2).sum(-1)
    sinr=sig/(tot-sig+nl.unsqueeze(1)+1e-10)
    return -torch.log2(1+sinr).sum(1).mean()

@torch.no_grad()
def evaluate_model(model,loader,device):
    model.eval(); total,n=0.0,0
    for Hh,H,snr,_ in loader:
        Hh,H,snr=Hh.to(device),H.to(device),snr.to(device)
        loss=sinr_sumrate_loss(model(Hh,snr),H,snr,model.Nt,model.K)
        total+=(-loss.item())*Hh.shape[0]; n+=Hh.shape[0]
    model.train(); return total/n


# ── Generic complex-channel helpers used by the faithful baselines below ──
def flat_to_H(H_flat, K, Nt):
    B = H_flat.shape[0]
    half = K * Nt
    return torch.complex(H_flat[:, :half].view(B, K, Nt), H_flat[:, half:].view(B, K, Nt))

def W_to_flat(W):
    """W: (B, Nt, K) complex -> (B, 2*Nt*K) real."""
    B = W.shape[0]
    return torch.cat([W.real.reshape(B, -1), W.imag.reshape(B, -1)], dim=1)

def mmse_bf_flat(H_flat, snr_db, K, Nt, P_total=1.0):
    """LEGACY / unused for pretraining now (see train_model, which uses the
    offline WMMSE target from the dataloader instead) -- kept only because
    a couple of baselines below still expose _mmse_from_flat for reference."""
    B = H_flat.shape[0]
    H = flat_to_H(H_flat, K, Nt)
    sl = 10 ** (snr_db / 10)
    reg = K / sl
    eye = torch.eye(K, dtype=H.dtype, device=H.device).unsqueeze(0)
    HHH = H @ H.conj().transpose(-2, -1)
    try:
        W = H.conj().transpose(-2, -1) @ torch.linalg.inv(HHH + reg.view(B, 1, 1) * eye)
    except Exception:
        W = H.conj().transpose(-2, -1) @ torch.linalg.pinv(HHH + reg.view(B, 1, 1) * eye)
    n = torch.linalg.norm(W.reshape(B, -1), dim=1, keepdim=True).clamp(min=1e-8).unsqueeze(-1)
    W = W * (P_total ** 0.5) / n
    return W_to_flat(W)

def recover_W_from_pl(H, p, lam, sigma2, eps=1e-8):
    """
    Closed-form optimal MISO precoder structure shared by Xia et al. Eq.(24)/(28)
    and Zhang et al. Eq.(8):
        w_k = sqrt(p_k) * normalize( (sigma^2 I + sum_j lam_j h_j h_j^H)^-1 h_k )
    H: (B,K,Nt) complex, row k = h_k^H.  p, lam: (B,K) real, each summing to P_total.
    Returns W: (B,Nt,K) complex, column k = w_k.
    """
    B, K, Nt = H.shape
    Hc = H.conj()                                     # Hc[b,k,:] = h_k (as a plain vector)
    lam_c = lam.to(H.dtype)
    weighted_Hc = Hc * lam_c.unsqueeze(-1)             # (B,K,Nt)
    C = torch.einsum('bkn,bkm->bnm', weighted_Hc, H)   # sum_k lam_k * h_k h_k^H -> (B,Nt,Nt)
    eye = torch.eye(Nt, dtype=H.dtype, device=H.device).unsqueeze(0)
    C = C + sigma2.to(H.dtype).view(B, 1, 1) * eye
    rhs = Hc.permute(0, 2, 1)                          # (B,Nt,K), columns = h_k
    Wdir = torch.linalg.solve(C, rhs)                  # (B,Nt,K)
    norms = torch.linalg.norm(Wdir, dim=1, keepdim=True).clamp(min=eps)
    Wdir = Wdir / norms
    p_c = torch.sqrt(p.clamp(min=0)).to(H.dtype).unsqueeze(1)  # (B,1,K)
    return Wdir * p_c

def normalize_pl(raw, K, P_total=1.0, eps=1e-8):
    """Split a (B,2K) sigmoid output into (p, lam), each L1-normalized to sum
    P_total (implements the scaling layer, Xia Eq.(27))."""
    p_raw, lam_raw = raw[:, :K], raw[:, K:]
    p = P_total * p_raw / p_raw.sum(1, keepdim=True).clamp(min=eps)
    lam = P_total * lam_raw / lam_raw.sum(1, keepdim=True).clamp(min=eps)
    return p, lam


# ══════════════════════════════════════════════════════════════════════
# BeamTransformer (Pre-LN + SNR token)
# ══════════════════════════════════════════════════════════════════════
class BeamNet(nn.Module):
    """
    Channel-token Transformer beamformer.
    Large : d_model=256, n_layers=6, n_heads=4  → 4.77 M params
    Small : d_model=128, n_layers=5, n_heads=4  → 1.01 M params

    Pretraining: yes -- MSE warm-start against the offline WMMSE target
    shipped by the dataloader (see train_model / PRETRAINED_CHECKPOINTS in
    the Configuration section), same recipe as beamforming_project_colab_v4_
    WMMSE.ipynb.
    """
    def __init__(self,Nt=32,K=8,d_model=256,n_layers=6,n_heads=4,dropout=0.1):
        super().__init__()
        self.Nt,self.K,self.d_model,self.n_layers=Nt,K,d_model,n_layers
        self.token_embed=nn.Linear(2*Nt,d_model)
        self.snr_embed=nn.Linear(1,d_model)
        enc=nn.TransformerEncoderLayer(d_model=d_model,nhead=n_heads,
            dim_feedforward=d_model*4,dropout=dropout,batch_first=True,norm_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=n_layers,norm=nn.LayerNorm(d_model))
        self.output_proj=nn.Linear(d_model,2*Nt)
        self.pnorm=PowerNorm()

    def forward(self,H_flat,snr_db):
        B,K,Nt=H_flat.shape[0],self.K,self.Nt; half=K*Nt
        Hr=H_flat[:,:half].view(B,K,Nt); Hi=H_flat[:,half:].view(B,K,Nt)
        tok=self.token_embed(torch.cat([Hr,Hi],dim=-1))
        snr_tok=self.snr_embed((snr_db/30).unsqueeze(1)).unsqueeze(1)
        out=self.transformer(torch.cat([snr_tok,tok],dim=1))[:,1:]
        w=self.output_proj(out)
        Wr=w[:,:,:Nt].permute(0,2,1).reshape(B,-1); Wi=w[:,:,Nt:].permute(0,2,1).reshape(B,-1)
        return self.pnorm(torch.cat([Wr,Wi],dim=1))

    def _mmse_from_flat(self,H_flat,snr_db):
        """LEGACY / unused: pretraining now uses the offline WMMSE target
        precomputed by wmmse_targets_batch and shipped via the dataloader
        (4th item per batch), not a per-model analytic target computed on
        the fly. Kept only for reference / potential MMSE-vs-WMMSE A/B."""
        B,Nt,K=H_flat.shape[0],self.Nt,self.K; half=K*Nt
        H=torch.complex(H_flat[:,:half].view(B,K,Nt),H_flat[:,half:].view(B,K,Nt))
        sl=10**(snr_db/10); reg=K/sl; eye=torch.eye(K,dtype=H.dtype,device=H.device).unsqueeze(0)
        try: W=H.conj().transpose(-2,-1)@torch.linalg.inv(H@H.conj().transpose(-2,-1)+reg.view(B,1,1)*eye)
        except: W=H.conj().transpose(-2,-1)@torch.linalg.pinv(H@H.conj().transpose(-2,-1)+reg.view(B,1,1)*eye)
        n=torch.linalg.norm(W.reshape(B,-1),dim=1,keepdim=True).clamp(min=1e-8).unsqueeze(-1)
        W=W/n; return torch.cat([W.real.reshape(B,-1),W.imag.reshape(B,-1)],dim=1)


# ══════════════════════════════════════════════════════════════════════
# BlackboxFNN  == XiaBNN: W. Xia et al., "A Deep Learning Framework for
# Optimization of MISO Downlink Beamforming," IEEE Trans. Commun.,
# vol.68, no.3, 2020.
# ══════════════════════════════════════════════════════════════════════
class XiaBNN(nn.Module):
    """
    CNN predicts key features (p, lambda) in R^{2K}; W is recovered via the
    closed-form optimal-structure formula -- never predicted directly.
    Architecture follows the paper's Table I: 2 conv layers (8 kernels, 3x3,
    pad 1) + BN + ReLU, flatten, FC -> 2K, sigmoid. Input is the I/Q-stacked
    channel as a (2, K*Nt) "image" (paper Section III-A.1).

    Pretraining: yes -- the paper trains this model with a supervised
    WMMSE-imitation stage before the sum-rate objective. Here that stage
    uses the offline WMMSE target shipped by the dataloader (see
    train_model), not the per-model _mmse_from_flat below (kept only as a
    LEGACY / unused reference, same as BeamNet's).

    Disclosed deviations: (1) a small SNR embedding is concatenated before the
    FC layer, since this study trains one model across a SNR range rather than
    Xia's per-fixed-config models; (2) the closed-form recovery step (Eq. 24)
    and the (p, lambda) prediction head are implemented exactly as specified.
    """
    def __init__(self, Nt=32, K=8, n_kernels=8, P_total=1.0):
        super().__init__()
        self.Nt, self.K, self.P_total = Nt, K, P_total
        NK = Nt * K
        self.conv = nn.Sequential(
            nn.Conv2d(1, n_kernels, kernel_size=3, padding=1), nn.BatchNorm2d(n_kernels), nn.ReLU(True),
            nn.Conv2d(n_kernels, n_kernels, kernel_size=3, padding=1), nn.BatchNorm2d(n_kernels), nn.ReLU(True),
        )
        self.snr_embed = nn.Linear(1, 32)
        self.fc = nn.Linear(n_kernels * 2 * NK + 32, 2 * K)
        self.pnorm = PowerNorm(P_total)

    def forward(self, H_flat, snr_db):
        B = H_flat.shape[0]
        H = flat_to_H(H_flat, self.K, self.Nt)
        img = torch.stack([H.real, H.imag], dim=1).reshape(B, 1, 2, self.K * self.Nt)
        feat = self.conv(img).reshape(B, -1)
        s = self.snr_embed((snr_db / 30.0).unsqueeze(1))
        raw = torch.sigmoid(self.fc(torch.cat([feat, s], dim=1)))
        p, lam = normalize_pl(raw, self.K, self.P_total)
        sigma2 = 10.0 ** (-snr_db / 10.0)
        W = recover_W_from_pl(H, p, lam, sigma2)
        return self.pnorm(W_to_flat(W))

    def _mmse_from_flat(self, H_flat, snr_db):
        """LEGACY / unused -- see BeamNet._mmse_from_flat above."""
        return mmse_bf_flat(H_flat, snr_db, self.K, self.Nt, self.P_total)


# ══════════════════════════════════════════════════════════════════════
# IAIDNN  — Q. Hu et al., "Iterative Algorithm Induced Deep-Unfolding
# Neural Networks: Precoding Design for Multiuser MIMO Systems," IEEE
# Trans. Wireless Commun., vol.20, no.2, 2021. DOI: 10.1109/TWC.2020.3033334
# ══════════════════════════════════════════════════════════════════════
class IAIDNNLayer(nn.Module):
    """
    One unfolded WMMSE iteration (paper's Eqs. 15-16), specialized to the
    MISO case (N_r=1, d_k=1) so U_k, W_k collapse to scalars -- their update
    is then exact (no approximation needed for a 1x1 "inverse"). The genuine
    complexity bottleneck the paper targets is the N_t x N_t inversion in the
    V-update; that one is replaced with the paper's learnable approximation
    B^+ X_k + B Y_k + Z_k + O_k, where B^+ is the diagonal-reciprocal operator
    the paper defines (not a real matrix inverse) -- this is what avoids an
    O(Nt^3) matrix inversion at inference time.

    Disclosed deviation: standard autograd is used for the backward pass
    instead of the paper's closed-form "generalized chain rule" -- mathematically
    equivalent, just implemented differently.
    """
    def __init__(self, Nt, K, last_layer=False):
        super().__init__()
        self.Nt, self.K, self.last_layer = Nt, K, last_layer
        if not last_layer:
            # X_k = I, Y_k = Z_k = O_k = 0 at init, so the layer starts out as the
            # cheap diagonal-reciprocal (Jacobi-style) solve B^+ @ rhs -- a near-zero
            # random init here would make Vnew ~ 0 pre-normalization and destabilize
            # the per-layer unit-power renormalization below.
            eye_k = torch.eye(Nt).unsqueeze(0).repeat(K, 1, 1)
            self.Xr = nn.Parameter(eye_k.clone())
            self.Xi = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Yr = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Yi = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Zr = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Zi = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Or = nn.Parameter(torch.zeros(K, Nt))
            self.Oi = nn.Parameter(torch.zeros(K, Nt))

    def forward(self, V, H, sigma2, eps=1e-8):
        B, K, Nt = H.shape

        # ---- U, W updates: exact scalars (Nr=1, d_k=1 special case) ----
        HV = torch.einsum('bkn,bnm->bkm', H, V)
        HVkk = torch.diagonal(HV, dim1=1, dim2=2)
        Vpow = (V.abs() ** 2).sum(dim=(1, 2))
        A = sigma2.unsqueeze(1) * Vpow.unsqueeze(1) + (HV.abs() ** 2).sum(dim=2)
        U = HVkk / (A.to(H.dtype) + eps)
        E = 1.0 - U.conj() * HVkk
        Wsc = 1.0 / (E + eps)
        Wsc_r = Wsc.real.clamp(min=eps)

        # ---- B matrix (Nt x Nt), the actual complexity bottleneck ----
        coef = (U.abs() ** 2) * Wsc_r
        diag_term = (sigma2 * coef.sum(dim=1)).to(H.dtype)
        Hc = H.conj()
        weighted_Hc = Hc * coef.to(H.dtype).unsqueeze(-1)
        outer = torch.einsum('bkn,bkm->bnm', weighted_Hc, H)
        eye = torch.eye(Nt, dtype=H.dtype, device=H.device).unsqueeze(0)
        Bmat = diag_term.view(-1, 1, 1) * eye + outer

        rhs = Hc * (U * Wsc_r.to(H.dtype)).unsqueeze(-1)
        rhs = rhs.permute(0, 2, 1)                            # (B, Nt, K)

        if self.last_layer:
            Vnew = torch.linalg.solve(Bmat, rhs)              # exact closed-form final refinement
        else:
            # Apply X_k, Y_k, Z_k, B, B+ to the RHS VECTOR at each step (O(Nt^2) each),
            # never forming the full Nt x Nt operator T_k explicitly (that would cost two
            # O(Nt^3) matrix-matrix products per user for no numerical benefit). Vectorized
            # over users (no Python loop): K is treated as a batched dimension alongside
            # the sample batch B, so all users' matrix-vector products run as a handful of
            # batched einsum calls instead of K sequential per-layer kernel launches -- a
            # per-user Python loop here is overhead-bound on GPU, not compute-bound,
            # despite the modest FLOP count. Arithmetically identical to looping over k
            # (verified bit-for-bit).
            Bdiag = torch.diagonal(Bmat, dim1=1, dim2=2)
            Bplus = torch.diag_embed(1.0 / (Bdiag + eps))
            X = torch.complex(self.Xr, self.Xi)               # (K,Nt,Nt)
            Y = torch.complex(self.Yr, self.Yi)
            Z = torch.complex(self.Zr, self.Zi)
            O = torch.complex(self.Or, self.Oi)                # (K,Nt)
            r = rhs.permute(0, 2, 1)                           # (B,K,Nt)
            Xr_ = torch.einsum('knm,bkm->bkn', X, r)
            Yr_ = torch.einsum('knm,bkm->bkn', Y, r)
            Zr_ = torch.einsum('knm,bkm->bkn', Z, r)
            t1  = torch.einsum('bnm,bkm->bkn', Bplus, Xr_)
            t2  = torch.einsum('bnm,bkm->bkn', Bmat, Yr_)
            Vnew = (t1 + t2 + Zr_ + O.unsqueeze(0)).permute(0, 2, 1)   # (B,Nt,K)

        Vnew = Vnew / Vnew.norm(dim=(1, 2), keepdim=True).clamp(min=eps)
        return Vnew


class IAIDNN(nn.Module):
    """
    Deep-unfolded WMMSE with L learnable iterations.
    Ref: Q. Hu et al., IEEE Trans. Wireless Commun., vol.20, no.2, 2021.
         DOI: 10.1109/TWC.2020.3033334

    Pretraining: NO. Per the paper, IAIDNN is trained end-to-end on the
    unsupervised sum-rate loss from the start -- there is no supervised
    warm-start stage. Structural initialisation substitutes for it: V^0
    below is a zero-forcing-style projection, and each IAIDNNLayer is
    itself one learnable WMMSE iteration, so the network already starts
    inside WMMSE's basin rather than needing a label-matching phase first.
    (It's the paper's separate blackbox CNN benchmark -- i.e. BlackboxFNN/
    XiaBNN above -- that pretrains on WMMSE labels, not IAIDNN.) See
    build_all_models / the training loop, which sets pretrain_epochs=0
    specifically for this model.
    """
    def __init__(self, Nt=32, K=8, n_layers=5, P_total=1.0):
        super().__init__()
        self.Nt, self.K, self.P_total = Nt, K, P_total
        self.layers = nn.ModuleList(
            [IAIDNNLayer(Nt, K, last_layer=(i == n_layers - 1)) for i in range(n_layers)]
        )
        self.pnorm = PowerNorm(P_total)

    def forward(self, H_flat, snr_db):
        B = H_flat.shape[0]
        H = flat_to_H(H_flat, self.K, self.Nt)
        sigma2 = 10.0 ** (-snr_db / 10.0)
        # Zero-forcing initialization ("V_k^0 initialized using the zero-forcing precoder")
        HHc = H.conj().transpose(-2, -1) @ H
        eye = torch.eye(self.Nt, dtype=HHc.dtype, device=HHc.device).unsqueeze(0)
        V = torch.linalg.solve(HHc + 1e-6 * eye, H.conj().transpose(-2, -1))
        V = V / V.norm(dim=(1, 2), keepdim=True).clamp(min=1e-8)
        for layer in self.layers:
            V = layer(V, H, sigma2)
        return self.pnorm(W_to_flat(V))

    def _mmse_from_flat(self, H_flat, snr_db):
        """LEGACY / unused: IAIDNN never pretrains (see class docstring),
        so this is never actually called -- kept only for reference."""
        return mmse_bf_flat(H_flat, snr_db, self.K, self.Nt, self.P_total)


# ══════════════════════════════════════════════════════════════════════
# FNN-Zhang  == ZhangFNN: M. Zhang, J. Gao & C. Zhong, "A Deep Learning-
# Based Framework for Low Complexity Multiuser MIMO Precoding Design,"
# IEEE Trans. Wireless Commun., vol.21, no.12, 2022. DOI: 10.1109/TWC.2022.3198060
# ══════════════════════════════════════════════════════════════════════
class ZhangFNN(nn.Module):
    """
    Same (p, lambda) + closed-form recovery as XiaBNN, but the network sees
    the compressed channel correlation (Gram) matrix G = H H^H (K x K) instead
    of the raw NK-dim channel -- the paper's core efficiency idea (Section
    IV-A1: "we just need the inner product of each user channel vector").
    Backbone is Conv+BN+LeakyReLU (paper's IV-A2) sized for the much smaller
    K x K input.

    Pretraining: yes -- the paper explicitly trains "by combining supervised
    and unsupervised learning": a supervised stage with WMMSE-produced labels
    and an MSE loss, then unsupervised sum-rate fine-tuning. Here that stage
    uses the offline WMMSE target shipped by the dataloader (see train_model),
    not the per-model _mmse_from_flat below (LEGACY / unused, same as
    BeamNet's / XiaBNN's).

    Disclosed deviations: SNR embedding and unified training protocol as for
    XiaBNN above; the paper's additional recovery-module compression / network
    pruning (Section IV-B, an efficiency add-on orthogonal to correctness) is
    not implemented.
    """
    def __init__(self, Nt=32, K=8, n_kernels=16, P_total=1.0):
        super().__init__()
        self.Nt, self.K, self.P_total = Nt, K, P_total
        self.conv = nn.Sequential(
            nn.Conv2d(1, n_kernels, kernel_size=3, padding=1), nn.BatchNorm2d(n_kernels), nn.LeakyReLU(0.1, True),
            nn.Conv2d(n_kernels, n_kernels, kernel_size=3, padding=1), nn.BatchNorm2d(n_kernels), nn.LeakyReLU(0.1, True),
        )
        self.snr_embed = nn.Linear(1, 32)
        self.fc = nn.Linear(n_kernels * 2 * K * K + 32, 2 * K)
        self.pnorm = PowerNorm(P_total)

    def forward(self, H_flat, snr_db):
        B = H_flat.shape[0]
        H = flat_to_H(H_flat, self.K, self.Nt)
        G = torch.matmul(H, H.conj().transpose(-2, -1))     # (B,K,K) Gram matrix h_i^H h_j
        img = torch.stack([G.real, G.imag], dim=1).reshape(B, 1, 2 * self.K, self.K)
        feat = self.conv(img).reshape(B, -1)
        s = self.snr_embed((snr_db / 30.0).unsqueeze(1))
        raw = torch.sigmoid(self.fc(torch.cat([feat, s], dim=1)))
        p, lam = normalize_pl(raw, self.K, self.P_total)
        sigma2 = 10.0 ** (-snr_db / 10.0)
        W = recover_W_from_pl(H, p, lam, sigma2)
        return self.pnorm(W_to_flat(W))

    def _mmse_from_flat(self, H_flat, snr_db):
        """LEGACY / unused -- see BeamNet._mmse_from_flat above."""
        return mmse_bf_flat(H_flat, snr_db, self.K, self.Nt, self.P_total)


# ── Model registry ────────────────────────────────────────────────────
def build_all_models(Nt, K):
    # BeamNet-Large intentionally excluded from this comparison run.
    # Dict keys unchanged from before (BlackboxFNN / FNN-Zhang) even though
    # the underlying classes are now the paper-faithful XiaBNN / ZhangFNN --
    # every other cell in this notebook (plots, docx report, Config flags)
    # references models by these string keys, not by Python class name.
    return {
        'BlackboxFNN':   XiaBNN(Nt=Nt, K=K),
        'IAIDNN':        IAIDNN(Nt=Nt, K=K, n_layers=5),
        'FNN-Zhang':     ZhangFNN(Nt=Nt, K=K),
        'BeamNet-Small': BeamNet(Nt=Nt, K=K, d_model=128, n_layers=5, n_heads=4),
    }
def model_params(model): return sum(p.numel() for p in model.parameters())

print('Model parameter counts (Nt=32, K=8):')
for name,m in build_all_models(32,8).items():
    print(f'  {name:<18}: {model_params(m):>8,}')


## 6 — Training infrastructure

In [ ]:
def train_model(model, name, train_loader, val_loader,
                device, lr, max_epochs, patience,
                pretrain_epochs=0, grad_clip=1.0):
    """
    Universal trainer for all models.
    Every model gets MSE pre-training against the offline WMMSE target
    shipped with the dataset (4th item yielded by the loader), then
    end-to-end sum-rate fine-tuning.
    """
    model=model.to(device)
    opt=optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    sch=ReduceLROnPlateau(opt,mode='max',factor=0.5,patience=5,min_lr=1e-6)
    mse_loss=nn.MSELoss()
    best_val,no_impr=-float('inf'),0; history=[]
    total_p=model_params(model)
    print(f'\n[Train] {name}  ({total_p:,} params)')

    # ── Pre-training (ALL models, including baselines) ─────────────────
    # The offline WMMSE target is precomputed and shipped by the dataloader
    # (4th item each batch yields), so pretraining a model against it needs
    # nothing model-specific anymore -- just MSE(model(Hh,snr), W_wmmse).
    # No architecture opt-in required, unlike the old _mmse_from_flat scheme.
    if pretrain_epochs>0:
        print(f'  Pre-training {pretrain_epochs} epochs (MSE -> WMMSE target) ...')
        pre_opt=optim.Adam(model.parameters(),lr=1e-3)
        for ep in range(1,pretrain_epochs+1):
            model.train(); tl,n=0.0,0
            for Hh,_,snr,W_wmmse in train_loader:
                Hh,snr,W_wmmse=Hh.to(device),snr.to(device),W_wmmse.to(device)
                pre_opt.zero_grad()
                loss=mse_loss(model(Hh,snr),W_wmmse)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),grad_clip)
                pre_opt.step(); tl+=loss.item()*Hh.shape[0]; n+=Hh.shape[0]
            print(f'    Pre-train {ep:>2}/{pretrain_epochs}  MSE={tl/n:.6f}')
        print('  Pre-training done.')

    # ── Sum-rate fine-tuning ──────────────────────────────────────────
    print(f'  {"Ep":>4}  {"TrainLoss":>10}  {"ValRate":>9}  {"LR":>9}  {"t":>6}')
    print('  '+'-'*48)
    for ep in range(1,max_epochs+1):
        if hasattr(train_loader.sampler,'set_epoch'):
            train_loader.sampler.set_epoch(ep)
        model.train(); tl,n=0.0,0; t0=time.time()
        for Hh,H,snr,_ in train_loader:
            Hh,H,snr=Hh.to(device),H.to(device),snr.to(device)
            opt.zero_grad()
            loss=sinr_sumrate_loss(model(Hh,snr),H,snr,model.Nt,model.K)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),grad_clip)
            opt.step(); tl+=loss.item()*Hh.shape[0]; n+=Hh.shape[0]
        tl/=n; vr=evaluate_model(model,val_loader,device); sch.step(vr)
        lr_now=opt.param_groups[0]['lr']
        print(f'  {ep:>4}  {tl:>10.4f}  {vr:>9.4f}  {lr_now:>9.2e}  {time.time()-t0:>5.1f}s')
        history.append({'epoch':ep,'train_loss':tl,'val_rate':vr})
        if vr>best_val:
            best_val,no_impr=vr,0
            torch.save({'state_dict':model.state_dict(),'epoch':ep,'val_rate':vr},
                       os.path.join(CHECKPOINT_DIR,f'{name}_best.pt'))
        else:
            no_impr+=1
            if no_impr>=patience:
                print(f'  [Early stop] no improvement for {patience} epochs.'); break

    # Save curve
    with open(os.path.join(RESULTS_DIR,f'{name}_curve.csv'),'w',newline='') as f:
        csv.DictWriter(f,['epoch','train_loss','val_rate']).writeheader()
        csv.DictWriter(f,['epoch','train_loss','val_rate']).writerows(history)
    print(f'  Best val = {best_val:.4f} bps/Hz')
    return history, best_val

def load_model(model, name, path=None):
    """
    Loads a checkpoint into `model`. If `path` is given, loads from that
    exact file (e.g. a checkpoint saved by a previous run of this notebook,
    or by beamforming_project_colab_v4_WMMSE.ipynb — see
    PRETRAINED_CHECKPOINTS in the Configuration section). Otherwise falls
    back to this run's own CHECKPOINT_DIR/{name}_best.pt.
    """
    p = path or os.path.join(CHECKPOINT_DIR, f'{name}_best.pt')
    if os.path.exists(p):
        ck = torch.load(p, map_location='cpu')
        model.load_state_dict(ck['state_dict'])
        print(f'[Load] {name}  epoch={ck.get("epoch","?")}  '
              f'val={ck.get("val_rate", float("nan")):.4f}  from {p}')
    else:
        print(f'[Load] WARNING: no checkpoint found for {name} at {p}')
    return model


## 7 — Configuration

In [ ]:
# ── System ────────────────────────────────────────────────────────────
# Set to match the ACTUAL config you ran in beamforming_project_colab_v4_
# WMMSE.ipynb (Nt=32, K=8, L=5, D_MODEL=128, N_LAYERS=5, N_HEADS=4 -- this
# is exactly BeamNet-Small's architecture). get_dataloaders() hashes these
# values, plus dataset sizes / SNR ranges / WMMSE settings, to find the
# cache directory to reuse -- if they don't match what v4 actually used,
# the cache just won't be found and this notebook regenerates its own
# (still correct, just slower on the first run).
Nt = 32
K  = 8
L  = 5
ARRAY = 'ULA'

# ── Dataset ───────────────────────────────────────────────────────────
# Also must match v4's dataset sizes for the cache hash to line up.
N_TRAIN    = 200_000
N_VAL      = 10_000
N_TEST     = 20_000
BATCH_SIZE = 256
CHUNK_SIZE = 10_000

# ── WMMSE pre-training labels (only matters if the cache above is missed
# and this notebook has to build its own dataset from scratch) ─────────
WMMSE_N_ITER = 30
WMMSE_TOL    = 1e-4

# ── Dataset caching ───────────────────────────────────────────────────
USE_DATASET_CACHE     = True   # reuse V4_DATASET_CACHE_DIR (see Drive-mount cell) if it matches
FORCE_REGENERATE_DATA = False  # True -> ignore any existing cache and rebuild

# ── Training ──────────────────────────────────────────────────────────
LR              = 1e-4
MAX_EPOCHS      = 80
PATIENCE        = 25
PRETRAIN_EPOCHS = 20   # WMMSE MSE pretraining -- applies to BeamNet-Small, BlackboxFNN,
                       # and FNN-Zhang (all three do this in their source papers). IAIDNN is
                       # deliberately excluded: per Hu et al. 2021, IAIDNN trains end-to-end on
                       # the unsupervised sum-rate loss from the start (see build/train loop).

# ── Reuse previously-trained checkpoints instead of retraining ─────────
# Set a path to REUSE that checkpoint (no training happens for that model).
# Leave None to train that model from scratch in this run.
#
# BeamNet-Small's architecture (d_model=128, n_layers=5, n_heads=4) and
# system dims (Nt=32, K=8) are exactly what beamforming_project_colab_v4_
# WMMSE.ipynb trained (per your config), so its saved checkpoint drops in
# here directly -- BeamNet-Large is skipped entirely for this run (see
# build_all_models), so there's nothing to reuse or train for it.
# Point this at YOUR v4 run folder (printed by v4's Drive-mount cell as
# RUN_DIR, e.g. '/content/drive/MyDrive/BeamTransformer/Run_3').
V4_BEAMNET_SMALL_CHECKPOINT = '/content/drive/MyDrive/BeamTransformer_Comparison/checkpoints/beamnet_best.pt'

PRETRAINED_CHECKPOINTS = {
    'BeamNet-Small': V4_BEAMNET_SMALL_CHECKPOINT,  # <- reuse v4's trained model, no retraining
    'BlackboxFNN':   None,
    'IAIDNN':        None,
    'FNN-Zhang':     None,
}
# Derived automatically: any model with a checkpoint path is skipped during training.
SKIP_TRAINING = {name: (path is not None) for name, path in PRETRAINED_CHECKPOINTS.items()}

print(f'Nt={Nt}, K={K}, L={L}, array={ARRAY}')
print(f'Train/Val/Test: {N_TRAIN}/{N_VAL}/{N_TEST}')
print(f'Device: {DEVICE}')
print(f'Skip training: {[k for k,v in SKIP_TRAINING.items() if v]}')


## 8 — Build dataset

In [ ]:
train_loader, val_loader, test_loader, channel, train_ds, train_sm = get_dataloaders(
    Nt=Nt, K=K, L=L, array_type=ARRAY,
    n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST,
    batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE,
    wmmse_n_iter=WMMSE_N_ITER, wmmse_tol=WMMSE_TOL,
    cache_dir=(V4_DATASET_CACHE_DIR if USE_DATASET_CACHE else None),
    force_regenerate=FORCE_REGENERATE_DATA,
)


## 9 — Train all models

In [ ]:
all_models   = build_all_models(Nt, K)
all_histories = {}
all_val_rates = {}

for name, model in all_models.items():
    ckpt_path = PRETRAINED_CHECKPOINTS.get(name)

    if SKIP_TRAINING.get(name, False):
        print(f'[Skip] {name} — reusing saved checkpoint (no training)')
        all_models[name] = load_model(model, name, path=ckpt_path).to(DEVICE)
        p = ckpt_path or os.path.join(CHECKPOINT_DIR, f'{name}_best.pt')
        if os.path.exists(p):
            ck = torch.load(p, map_location='cpu')
            all_val_rates[name] = ck.get('val_rate', float('nan'))
        else:
            all_val_rates[name] = float('nan')
        continue

    # WMMSE MSE pre-training applies to every model EXCEPT IAIDNN.
    # Per Hu et al. 2021, IAIDNN is trained end-to-end with the unsupervised
    # sum-rate loss from the start (ZF-initialised V^0, no supervised warm
    # start) -- it's the paper's own black-box CNN baseline that pretrains
    # against WMMSE labels, not IAIDNN itself. BlackboxFNN and FNN-Zhang's
    # papers both do use WMMSE-label pretraining, as does BeamNet-Small.
    model_pretrain_epochs = 0 if name == 'IAIDNN' else PRETRAIN_EPOCHS

    hist, best_val = train_model(
        model=model, name=name,
        train_loader=train_loader, val_loader=val_loader,
        device=DEVICE, lr=LR,
        max_epochs=MAX_EPOCHS, patience=PATIENCE,
        pretrain_epochs=model_pretrain_epochs,
    )
    all_histories[name] = hist
    all_val_rates[name] = best_val
    all_models[name]    = load_model(model, name).to(DEVICE)

# Free the local dataset cache if it was a throwaway temp dir; a no-op
# (with a printed message) if it's the shared/persistent Drive cache.
train_ds.cleanup()

print('\n[Done] All models ready (trained or reused).')
print(f'  {"Model":<18}  {"Val Sum-Rate":>14}  {"Params":>10}')
print('  '+'-'*46)
for name, model in all_models.items():
    print(f'  {name:<18}  {all_val_rates[name]:>12.4f} bps/Hz  {model_params(model):>10,}')


## 10 — Save run parameters

In [ ]:
params_lines = [
    f'Run ID          : {RUN_ID}',
    f'Timestamp       : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
    f'Device          : {DEVICE}',
    '',
    '=== System ===',
    f'Nt              : {Nt}',
    f'K               : {K}',
    f'L               : {L}',
    '',
    '=== Dataset ===',
    f'N_TRAIN         : {N_TRAIN}',
    f'N_VAL           : {N_VAL}',
    f'N_TEST          : {N_TEST}',
    f'BATCH_SIZE      : {BATCH_SIZE}',
    '',
    '=== Training ===',
    f'LR              : {LR}',
    f'MAX_EPOCHS      : {MAX_EPOCHS}',
    f'PATIENCE        : {PATIENCE}',
    f'PRETRAIN_EPOCHS : {PRETRAIN_EPOCHS}',
    '',
    '=== Val sum-rates ===',
] + [f'  {n:<18}: {v:.4f} bps/Hz' for n,v in all_val_rates.items()]

with open(os.path.join(RUN_DIR,'run_params.txt'),'w') as f:
    f.write('\n'.join(params_lines))
print('Saved run_params.txt')


## 11 — Evaluation suite

In [ ]:
# ── Plot style ────────────────────────────────────────────────────────
plt.rcParams.update({'font.size':11,'axes.labelsize':12,'legend.fontsize':9,
                     'lines.linewidth':2.0,'lines.markersize':5,
                     'axes.spines.top':False,'axes.spines.right':False,'grid.alpha':0.35})

# Colour/marker map covering all 5 models + 4 classical
CMAP = {
    'BeamNet-Small': '#1565c0',   # mid blue
    'BlackboxFNN':   '#e65100',   # orange
    'IAIDNN':        '#2e7d32',   # green
    'FNN-Zhang':     '#6a1b9a',   # purple
    'MRT':           '#9e9e9e',   # grey
    'ZF-BF':         '#f9a825',   # yellow
    'MMSE-BF':       '#c62828',   # red
    'WMMSE':         '#ad1457',   # magenta — near-optimal iterative reference
    'Perfect CSI':   '#00695c',   # teal dashed
}
MMAP = {
    'BeamNet-Small':'s',
    'BlackboxFNN':'^','IAIDNN':'D','FNN-Zhang':'P',
    'MRT':'x','ZF-BF':'+','MMSE-BF':'v','WMMSE':'<','Perfect CSI':'*',
}
LSTY = {'Perfect CSI':'--'}

def save_plot_data(fname, **arrays):
    d=os.path.join(RESULTS_DIR,'plot_data'); os.makedirs(d,exist_ok=True)
    np.savez(os.path.join(d,fname+'.npz'),**{k:np.asarray(v) for k,v in arrays.items()})
    try:
        import csv as _c; keys=list(arrays.keys()); vals=[np.asarray(v).flatten() for v in arrays.values()]
        nr=max(len(v) for v in vals)
        with open(os.path.join(d,fname+'.csv'),'w',newline='') as f:
            w=_c.writer(f); w.writerow(keys)
            for row in zip(*[v.tolist()+['']*(nr-len(v)) for v in vals]): w.writerow(row)
    except: pass
    print(f'  [Data] {fname}.npz/.csv')


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11a. Sum-Rate vs. SNR  (all 5 DL models + 4 classical)
# ══════════════════════════════════════════════════════════════════════

def eval_sumrate_all(models_dict, Nt, K, n_samples=1500,
                     snr_range=(-5,35), n_points=17, device='cpu'):
    snr_dbs=np.linspace(*snr_range,n_points)
    ch=MassiveMIMOChannel(Nt=Nt,K=K); rng=np.random.default_rng(5)
    H_true=ch.generate(n_samples,rng=rng); H_hat=add_csi_noise(H_true,15.0,rng=rng)
    results={'MRT':[],'ZF-BF':[],'MMSE-BF':[],'WMMSE':[],'Perfect CSI':[]}
    for n in models_dict: results[n]=[]

    for snr in snr_dbs:
        rates={k:[] for k in results}
        for i in range(n_samples):
            H,Hh=H_true[i],H_hat[i]
            rates['MRT']        .append(compute_sum_rate(H,mrt(Hh),snr))
            rates['ZF-BF']      .append(compute_sum_rate(H,zf_bf(Hh),snr))
            rates['MMSE-BF']    .append(compute_sum_rate(H,mmse_bf(Hh,snr),snr))
            rates['WMMSE']      .append(compute_sum_rate(H,wmmse_bf(Hh,snr),snr))
            rates['Perfect CSI'].append(compute_sum_rate(H,mmse_bf(H,snr),snr))
        Hr=torch.from_numpy(H_hat.real.reshape(n_samples,-1).astype(np.float32))
        Hi=torch.from_numpy(H_hat.imag.reshape(n_samples,-1).astype(np.float32))
        H_in=torch.cat([Hr,Hi],dim=1).to(device)
        snr_t=torch.full((n_samples,),snr,dtype=torch.float32).to(device)
        for mname,model in models_dict.items():
            model.eval()
            with torch.no_grad(): W_out=model(H_in,snr_t).cpu().numpy()
            rates[mname]=[compute_sum_rate(H_true[i],flat_to_complex_W(W_out[i],Nt,K),snr)
                          for i in range(n_samples)]
        for k in results: results[k].append(np.mean(rates[k]))
    return snr_dbs,results

print('[Eval] Sum-Rate vs SNR (all models) ...')
snr_dbs, sr_results = eval_sumrate_all(all_models, Nt, K, device=DEVICE)

fig,ax=plt.subplots(figsize=(8,5))
classical=['MRT','ZF-BF','MMSE-BF','WMMSE','Perfect CSI']
for label in classical:
    ax.plot(snr_dbs,sr_results[label],color=CMAP[label],
            marker=MMAP[label],markevery=3,linestyle=LSTY.get(label,'-'),
            alpha=0.7,label=label)
for label in all_models:
    ax.plot(snr_dbs,sr_results[label],color=CMAP[label],
            marker=MMAP[label],markevery=3,label=label,linewidth=2.2)
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Sum-Rate (bps/Hz)')
ax.set_title(f'Sum-Rate vs. SNR  —  Nt={Nt}, K={K}, pilot SNR=15 dB')
ax.legend(ncol=2,fontsize=9); ax.grid(True)
fig.tight_layout()
p=os.path.join(RESULTS_DIR,'sumrate_vs_snr_comparison.png')
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show(); print(f'[Plot] {p}')
save_plot_data('sumrate_vs_snr_comparison',snr_dbs=snr_dbs,
               **{k:np.array(v) for k,v in sr_results.items()})


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11b. BER vs. SNR
# ══════════════════════════════════════════════════════════════════════

def eval_ber_all(models_dict, Nt, K, n_ch=200,
                 snr_range=(-5,30), n_points=15, device='cpu'):
    snr_dbs=np.linspace(*snr_range,n_points)
    ch=MassiveMIMOChannel(Nt=Nt,K=K); rng=np.random.default_rng(7)
    H_true=ch.generate(n_ch,rng=rng); H_hat=add_csi_noise(H_true,15.0,rng=rng)
    results={k:[] for k in list(models_dict.keys())+['ZF-BF','MMSE-BF','WMMSE']}
    for snr in snr_dbs:
        rates={k:[] for k in results}
        for i in range(n_ch):
            H,Hh=H_true[i],H_hat[i]
            rates['ZF-BF']  .append(compute_ber(H,zf_bf(Hh),snr,rng=rng))
            rates['MMSE-BF'].append(compute_ber(H,mmse_bf(Hh,snr),snr,rng=rng))
            rates['WMMSE']  .append(compute_ber(H,wmmse_bf(Hh,snr),snr,rng=rng))
        Hr=torch.from_numpy(H_hat.real.reshape(n_ch,-1).astype(np.float32))
        Hi=torch.from_numpy(H_hat.imag.reshape(n_ch,-1).astype(np.float32))
        H_in=torch.cat([Hr,Hi],dim=1).to(device)
        snr_t=torch.full((n_ch,),snr,dtype=torch.float32).to(device)
        for mname,model in models_dict.items():
            model.eval()
            with torch.no_grad(): W_out=model(H_in,snr_t).cpu().numpy()
            rates[mname]=[compute_ber(H_true[i],flat_to_complex_W(W_out[i],Nt,K),snr,rng=rng)
                          for i in range(n_ch)]
        for k in results: results[k].append(np.mean(rates[k]))
    return snr_dbs,results

print('[Eval] BER vs SNR ...')
snr_dbs_ber,ber_results=eval_ber_all(all_models,Nt,K,device=DEVICE)

fig,ax=plt.subplots(figsize=(8,5))
for label in ['ZF-BF','MMSE-BF','WMMSE']:
    ax.semilogy(snr_dbs_ber,np.maximum(ber_results[label],1e-5),
                color=CMAP[label],marker=MMAP[label],markevery=3,alpha=0.7,label=label)
for label in all_models:
    ax.semilogy(snr_dbs_ber,np.maximum(ber_results[label],1e-5),
                color=CMAP[label],marker=MMAP[label],markevery=3,label=label,linewidth=2.2)
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('BER')
ax.set_title(f'BER vs. SNR  (QPSK, ZF-rx)  —  Nt={Nt}, K={K}')
ax.legend(ncol=2,fontsize=9); ax.grid(True,which='both')
fig.tight_layout()
p=os.path.join(RESULTS_DIR,'ber_vs_snr_comparison.png')
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show(); print(f'[Plot] {p}')
save_plot_data('ber_vs_snr_comparison',snr_dbs=snr_dbs_ber,
               **{k:np.array(v) for k,v in ber_results.items()})


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11c. CSI quality ablation  (sum-rate vs. pilot SNR)
# ══════════════════════════════════════════════════════════════════════

def eval_csi_quality_all(models_dict, Nt, K, n_samples=600,
                         data_snr=20.0, device='cpu'):
    pilot_snrs=np.linspace(0,20,11)
    ch=MassiveMIMOChannel(Nt=Nt,K=K); rng=np.random.default_rng(11)
    H_true=ch.generate(n_samples,rng=rng)
    results={k:[] for k in list(models_dict.keys())+['ZF-BF','MMSE-BF','WMMSE']}
    for p_snr in pilot_snrs:
        H_hat=add_csi_noise(H_true,p_snr,rng=rng)
        rates={k:[] for k in results}
        for i in range(n_samples):
            H,Hh=H_true[i],H_hat[i]
            rates['ZF-BF']  .append(compute_sum_rate(H,zf_bf(Hh),data_snr))
            rates['MMSE-BF'].append(compute_sum_rate(H,mmse_bf(Hh,data_snr),data_snr))
            rates['WMMSE']  .append(compute_sum_rate(H,wmmse_bf(Hh,data_snr),data_snr))
        Hr=torch.from_numpy(H_hat.real.reshape(n_samples,-1).astype(np.float32))
        Hi=torch.from_numpy(H_hat.imag.reshape(n_samples,-1).astype(np.float32))
        H_in=torch.cat([Hr,Hi],dim=1).to(device)
        snr_t=torch.full((n_samples,),data_snr,dtype=torch.float32).to(device)
        for mname,model in models_dict.items():
            model.eval()
            with torch.no_grad(): W_out=model(H_in,snr_t).cpu().numpy()
            rates[mname]=[compute_sum_rate(H_true[i],flat_to_complex_W(W_out[i],Nt,K),data_snr)
                          for i in range(n_samples)]
        for k in results: results[k].append(np.mean(rates[k]))
    return pilot_snrs,results

print('[Eval] CSI quality ablation ...')
pilot_snrs,csi_results=eval_csi_quality_all(all_models,Nt,K,device=DEVICE)

fig,ax=plt.subplots(figsize=(8,5))
for label in ['ZF-BF','MMSE-BF','WMMSE']:
    ax.plot(pilot_snrs,csi_results[label],color=CMAP[label],
            marker=MMAP[label],markevery=2,alpha=0.7,label=label)
for label in all_models:
    ax.plot(pilot_snrs,csi_results[label],color=CMAP[label],
            marker=MMAP[label],markevery=2,label=label,linewidth=2.2)
ax.set_xlabel('Pilot SNR (dB)'); ax.set_ylabel('Sum-Rate (bps/Hz)')
ax.set_title(f'Sum-Rate vs. CSI Quality  —  Nt={Nt}, K={K}, data SNR=20 dB')
ax.legend(ncol=2,fontsize=9); ax.grid(True)
fig.tight_layout()
p=os.path.join(RESULTS_DIR,'csi_quality_comparison.png')
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show(); print(f'[Plot] {p}')
save_plot_data('csi_quality_comparison',pilot_snrs=pilot_snrs,
               **{k:np.array(v) for k,v in csi_results.items()})


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11d. Complexity scatter: params vs. sum-rate @ 20 dB
# ══════════════════════════════════════════════════════════════════════
idx20=np.argmin(np.abs(snr_dbs-20))

fig,axes=plt.subplots(1,2,figsize=(13,5))

# Left: params vs sum-rate
ax=axes[0]
for label in all_models:
    p=model_params(all_models[label])/1e6
    r=sr_results[label][idx20]
    ax.scatter(p,r,color=CMAP[label],s=150,marker=MMAP[label],zorder=3)
    ax.annotate(label,(p,r),textcoords='offset points',xytext=(6,4),fontsize=9)
ax.set_xlabel('Parameters (M)'); ax.set_ylabel('Sum-Rate @ SNR=20 dB (bps/Hz)')
ax.set_title('Model Complexity vs. Performance'); ax.grid(True)

# Right: training curves (all 5 models)
ax=axes[1]
for name,hist in all_histories.items():
    eps=[h['epoch'] for h in hist]
    vrs=[h['val_rate'] for h in hist]
    ax.plot(eps,vrs,color=CMAP[name],label=name,linewidth=1.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Sum-Rate (bps/Hz)')
ax.set_title('Training Curves — All Models'); ax.legend(fontsize=9); ax.grid(True)

fig.tight_layout()
p=os.path.join(RESULTS_DIR,'complexity_and_curves.png')
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show(); print(f'[Plot] {p}')

# Summary table printed
print(f'\n{"Model":<18}  {"Params":>10}  {"Val (bps/Hz)":>14}  {"Test (bps/Hz)":>14}')
print('-'*62)


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11e. Test set evaluation for all models
# ══════════════════════════════════════════════════════════════════════
all_test_rates={}
for name,model in all_models.items():
    tr=evaluate_model(model,test_loader,DEVICE)
    all_test_rates[name]=tr

print(f'\n{"Model":<18}  {"Params":>10}  {"Val (bps/Hz)":>14}  {"Test (bps/Hz)":>14}')
print('-'*62)
for name,model in all_models.items():
    print(f'{name:<18}  {model_params(model):>10,}  '
          f'{all_val_rates[name]:>13.4f}  {all_test_rates[name]:>13.4f}')

# Save summary CSV
with open(os.path.join(RESULTS_DIR,'model_summary.csv'),'w',newline='') as f:
    w=csv.writer(f); w.writerow(['Model','Params','Val_bpsHz','Test_bpsHz'])
    for name,model in all_models.items():
        w.writerow([name,model_params(model),
                    f'{all_val_rates[name]:.4f}',f'{all_test_rates[name]:.4f}'])
print('\n[Data] model_summary.csv saved.')


## 12 — Generate Word report

In [ ]:
def _ensure_docx():
    if importlib.util.find_spec('docx') is None:
        subprocess.check_call([sys.executable,'-m','pip','install',
                               'python-docx','--break-system-packages','-q'])
_ensure_docx()
from docx import Document as _Doc
from docx.shared import Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

def _shd(cell,hex_c):
    tc=cell._tc; tcPr=tc.get_or_add_tcPr()
    s=OxmlElement('w:shd'); s.set(qn('w:val'),'clear')
    s.set(qn('w:color'),'auto'); s.set(qn('w:fill'),hex_c); tcPr.append(s)

def _hdr(table,texts,bg='1F3864'):
    for cell,t in zip(table.rows[0].cells,texts):
        cell.text=''; r=cell.paragraphs[0].add_run(t)
        r.bold=True; r.font.color.rgb=RGBColor(0xFF,0xFF,0xFF); r.font.size=Pt(9); _shd(cell,bg)

def _row(table,vals,bold0=False,alt=False):
    row=table.add_row()
    for j,v in enumerate(vals):
        c=row.cells[j]; c.text=str(v); c.paragraphs[0].runs[0].font.size=Pt(9)
        if bold0 and j==0: c.paragraphs[0].runs[0].bold=True
        if alt: _shd(c,'EBF3FB')

def _head(doc,text,level=1):
    p=doc.add_heading(text,level=level)
    for r in p.runs: r.font.color.rgb=RGBColor(0x1F,0x38,0x64)
    return p

def generate_comparison_report():
    doc=_Doc()
    for sec in doc.sections:
        sec.top_margin=Cm(2); sec.bottom_margin=Cm(2)
        sec.left_margin=Cm(2.5); sec.right_margin=Cm(2)

    t=doc.add_heading(f'BeamTransformer — Comparison Report (Run {RUN_ID})',0)
    t.alignment=WD_ALIGN_PARAGRAPH.CENTER
    for r in t.runs: r.font.color.rgb=RGBColor(0x1F,0x38,0x64)

    meta=[('Date',datetime.datetime.now().strftime('%d %B, %Y')),
          ('System',f'Nt={Nt}, K={K}, L={L}, ULA, pilot SNR=15 dB'),
          ('Dataset',f'{N_TRAIN}/{N_VAL}/{N_TEST} train/val/test'),
          ('Training',f'AdamW lr={LR}, max_epochs={MAX_EPOCHS}, patience={PATIENCE}'),
          ('Device',DEVICE)]
    tbl=doc.add_table(rows=len(meta),cols=2); tbl.style='Table Grid'
    for i,(k,v) in enumerate(meta):
        tbl.rows[i].cells[0].text=k; tbl.rows[i].cells[1].text=v
        tbl.rows[i].cells[0].paragraphs[0].runs[0].bold=True
        tbl.rows[i].cells[0].paragraphs[0].runs[0].font.size=Pt(10)
        tbl.rows[i].cells[1].paragraphs[0].runs[0].font.size=Pt(10)
        _shd(tbl.rows[i].cells[0],'D6E4F0')
    doc.add_paragraph()

    # 1. Model summary
    _head(doc,'1. Model Summary')
    t2=doc.add_table(rows=1,cols=4); t2.style='Table Grid'
    _hdr(t2,['Model','Parameters','Val Sum-Rate (bps/Hz)','Test Sum-Rate (bps/Hz)'])
    refs={'BeamNet-Small':'Proposed',
          'BlackboxFNN':'Xia et al. 2020 [B1]','IAIDNN':'Hu et al. 2021 [B2]',
          'FNN-Zhang':'Zhang et al. 2022 [B3]'}
    for i,(name,model) in enumerate(all_models.items()):
        _row(t2,[refs.get(name,name),f'{model_params(model):,}',
                 f'{all_val_rates[name]:.4f}',f'{all_test_rates[name]:.4f}'],
             bold0=True,alt=(i%2==1))
    doc.add_paragraph()


    # 2. Sum-rate at key SNR points
    _head(doc,'2. Sum-Rate at Key SNR Points')
    key_snrs=[-5,0,5,10,15,20,25,30,35]
    key_idx=[np.argmin(np.abs(snr_dbs-s)) for s in key_snrs]
    all_labels=['MRT','ZF-BF','MMSE-BF','WMMSE']+list(all_models.keys())+['Perfect CSI']
    cols=['SNR (dB)']+[l for l in all_labels if l in sr_results]
    t4=doc.add_table(rows=1,cols=len(cols)); t4.style='Table Grid'
    _hdr(t4,cols)
    for ii,idx in enumerate(key_idx):
        vals=[f'{key_snrs[ii]} dB']+[f'{sr_results[l][idx]:.3f}' for l in cols[1:]]
        _row(t4,vals,bold0=True,alt=(ii%2==1))
    doc.add_paragraph()

    # 3. Gain over MMSE-BF at 20 dB
    _head(doc,'3. Gain over MMSE-BF at SNR = 20 dB')
    mmse_ref=sr_results['MMSE-BF'][idx20]
    t5=doc.add_table(rows=1,cols=3); t5.style='Table Grid'
    _hdr(t5,['Model','Sum-Rate (bps/Hz)','Gain vs MMSE-BF'])
    dl_labels=['BlackboxFNN','IAIDNN','FNN-Zhang','BeamNet-Small','WMMSE']
    for i,lbl in enumerate(dl_labels):
        if lbl not in sr_results: continue
        r=sr_results[lbl][idx20]; diff=r-mmse_ref
        _row(t5,[lbl,f'{r:.4f}',f'{diff:+.4f}  ({diff/mmse_ref*100:+.1f}%)'],
             bold0=True,alt=(i%2==1))
    doc.add_paragraph()

    # 4. References
    _head(doc,'4. References')
    refs_text=[
        '[B1] W. Xia, G. Zheng, Y. Zhu, J. Zhang, J. Wang, and A. P. Petropulu, '
        '"A deep learning framework for optimization of MISO downlink beamforming," '
        'IEEE Trans. Commun., vol. 68, no. 3, pp. 1866-1880, Mar. 2020.',
        '[B2] Q. Hu, Y. Cai, Q. Shi, K. Xu, G. Yu, and Z. Ding, '
        '"Iterative algorithm induced deep-unfolding neural networks: Precoding design '
        'for multiuser MIMO systems," IEEE Trans. Wireless Commun., vol. 20, no. 2, '
        'pp. 1394-1410, Feb. 2021. DOI: 10.1109/TWC.2020.3033334',
        '[B3] M. Zhang, J. Gao, and C. Zhong, '
        '"A deep learning-based framework for low complexity multiuser MIMO precoding design," '
        'IEEE Trans. Wireless Commun., vol. 21, no. 12, pp. 11193-11206, Dec. 2022. '
        'DOI: 10.1109/TWC.2022.3198060',
    ]
    for ref in refs_text:
        p=doc.add_paragraph(ref); p.runs[0].font.size=Pt(9)

    path=os.path.join(RUN_DIR,f'Run_{RUN_ID}_Comparison_Report.docx')
    doc.save(path); print(f'[Report] {path}')
    return path

report_path=generate_comparison_report()


## 13 — Display plots

In [ ]:
from IPython.display import Image, display

plots=[
    f'{RESULTS_DIR}/sumrate_vs_snr_comparison.png',
    f'{RESULTS_DIR}/ber_vs_snr_comparison.png',
    f'{RESULTS_DIR}/csi_quality_comparison.png',
    f'{RESULTS_DIR}/complexity_and_curves.png',
]
for p in plots:
    if os.path.exists(p):
        print(f'\n── {os.path.basename(p)} ──')
        display(Image(filename=p))
    else:
        print(f'[Missing] {p}')

print(f'\n[Run {RUN_ID}] All results saved to: {RUN_DIR}')
print(f'  Plots      : {RESULTS_DIR}/')
print(f'  Plot data  : {RESULTS_DIR}/plot_data/')
print(f'  Checkpoints: {CHECKPOINT_DIR}/')
print(f'  Report     : {report_path}')
